# 02 - Supervised Policy and Value Training

Acquire labelled positions, train the shared network, and persist validation-selected checkpoints. Plots use completed epochs; no example scores are pre-filled.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT, "strategy.yaml")
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Training Device

Select a GPU runtime in Colab. CPU remains an explicit portability option. The default has six residual blocks and 128 channels.

In [ ]:
import torch
from chess_rl.reproducibility import resolve_device, seed_all
from chess_rl.model import ChessPolicyValueNetSmall

device = resolve_device(cfg["device"])
seed_all(cfg["seed"], cfg["deterministic"])
model_preview = ChessPolicyValueNetSmall(**cfg["model"])
print("Training device:", device)
print("Parameters:", sum(p.numel() for p in model_preview.parameters()))
print("Architecture:", model_preview.architecture)
del model_preview

## Offline Teacher Setup

Stockfish labels training data only. Default: 50,000 nodes, one thread, 128 MB hash. It is excluded from packaging. An explicit teacher: classical configuration is also supported.

In [ ]:
teacher_path = Path(cfg["dataset"]["engine_path"])
if cfg["dataset"]["teacher"] == "stockfish" and not teacher_path.is_file():
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "stockfish"])
if cfg["dataset"]["teacher"] == "stockfish":
    if not teacher_path.is_file():
        raise FileNotFoundError(f"Set dataset.engine_path to installed Stockfish: {teacher_path}")
    print("Teacher executable:", teacher_path)
    print("Teacher hash:", sha256(teacher_path))

## Prepare the Dataset

Supply compatible JSONL, such as converted Lichess Eval DB records, through dataset.jsonl_paths, or PGNs through dataset.pgn_paths for teacher labelling. Empty broad sources now stop with a clear error instead of generating synthetic training positions. The target is 100,000 unique positions. Saved annotation shards support resuming PGN teacher labelling.

In [ ]:
from chess_rl.dataset import prepare_dataset
dataset_manifest = prepare_dataset(PROJECT_ROOT, cfg)
print("Actual split counts:", dataset_manifest["counts"])
print("Requested target reached:", dataset_manifest["target_reached"])
print("Dataset hashes:", dataset_manifest["hashes"])

## Loss and Value Meaning

Policy loss uses legal moves. Outcome values use the side to move: win +1, draw 0, loss -1. Engine centipawns use tanh(cp/600), identified separately as heuristic labels. Validation loss is unsmoothed for comparable trials.

## Train or Resume

This runs the configured experiment. Latest resumes completed epochs; best_validation selects lowest validation loss. A partial interrupted epoch restarts from its last complete checkpoint.

In [ ]:
from chess_rl.training import fit_supervised
selected_pretraining = fit_supervised(PROJECT_ROOT, cfg, dataset_manifest)
print("Selected supervised checkpoint:", selected_pretraining)

## Learning Curves

Accuracy is against legal teacher targets. Lower loss does not itself prove stronger chess.

In [ ]:
from chess_rl.plots import plot_supervised
display(plot_supervised(PROJECT_ROOT, cfg["run_id"]))

## Optional Model Search

Disabled by default. Trials have separate folders. Three lowest-loss trials play development matches against the preserved classical agent; their playing scores select the initialization.

In [ ]:
RUN_MODEL_SEARCH = False
if RUN_MODEL_SEARCH:
    import yaml
    from chess_rl.tuning import run_model_study
    from chess_rl.evaluation import build_research_agent, run_matchup
    from chess_rl.reservations import load_reservations
    spaces = yaml.safe_load((PROJECT_ROOT / "configs/search_spaces.yaml").read_text())
    shortlist = run_model_study(PROJECT_ROOT, cfg, dataset_manifest, spaces)
    comparisons = []
    for trial in shortlist:
        checkpoint = PROJECT_ROOT / trial["checkpoint"]
        candidate = build_research_agent(PROJECT_ROOT, checkpoint, trial["config"], "shortlist")
        result = run_matchup(PROJECT_ROOT, candidate, PROJECT_ROOT / "reference/classical_agent",
                            PROJECT_ROOT / load_reservations(PROJECT_ROOT, cfg)["suites"]["development"],
                            trial["config"], "shortlist-development")
        comparisons.append((result["score"] if result["score"] is not None else -1, checkpoint))
    if not comparisons:
        raise RuntimeError("No successful model-search trials")
    selected_pretraining = max(comparisons, key=lambda item: item[0])[1]
    print("Shortlist match selection:", selected_pretraining)

## Persist Initialization Choice

Notebook 03b and optional stage 04a read this checkpoint reference. Independent experiments need different run identifiers.

In [ ]:
atomic_json(PROJECT_ROOT / "results" / cfg["run_id"] / "initial_selection.json",
            {"checkpoint": str(selected_pretraining.relative_to(PROJECT_ROOT)),
             "sha256": sha256(selected_pretraining)})
print("Notebook 02 complete. Open 03a_build_strategy_datasets.ipynb.")